In [0]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import mlflow
import mlflow.sklearn

from datetime import datetime

print("Libraries loaded successfully")

In [0]:
!pip install xgboost -q
%pip install lightgbm

In [0]:
DATA_PATH = "demand_forecasting.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

In [0]:
display(
    pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str),
        "missing_values": df.isnull().sum().values,
        "unique_values": [
            df[c].nunique() for c in df.columns
        ]
    })
)

In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

display(df.head())

In [0]:
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())

print(
    "Number of days:",
    (df["Date"].max() - df["Date"].min()).days + 1
)

In [0]:
print("Stores:", df["Store ID"].nunique())
print("Products:", df["Product ID"].nunique())
print("Categories:", df["Category"].nunique())
print("Regions:", df["Region"].nunique())

In [0]:
display(
    df.groupby(["Store ID", "Product ID"])
      .size()
      .describe()
)

In [0]:
display(df["Demand"].describe())

In [0]:
plt.figure(figsize=(10, 5))

sns.histplot(
    df["Demand"],
    bins=50,
    kde=True
)

plt.title("Demand Distribution")
plt.xlabel("Demand")
plt.ylabel("Frequency")

plt.show()

In [0]:
daily_demand = (
    df.groupby("Date")["Demand"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(16, 6))

plt.plot(
    daily_demand["Date"],
    daily_demand["Demand"]
)

plt.title("Total Daily Demand")
plt.xlabel("Date")
plt.ylabel("Demand")

plt.xticks(rotation=45)

plt.show()

In [0]:
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_values")
)

In [0]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [0]:
zero_demand_count = (df["Demand"] == 0).sum()

print("Rows with zero demand:", zero_demand_count)

print("Percentage of zero demand:",round(zero_demand_count / len(df) * 100,2),"%")


In [0]:
numeric_columns = df.select_dtypes(include=np.number).columns

display(df[numeric_columns].corr())


In [0]:
demand_correlation = (
    df[numeric_columns]
    .corr()["Demand"]
    .sort_values(ascending=False)
)

display(demand_correlation)


In [0]:
category_demand = (
    df.groupby("Category")["Demand"]
      .agg(["mean", "sum", "count"])
      .sort_values("mean", ascending=False)
      .reset_index()
)

display(category_demand)

In [0]:
region_demand = (
    df.groupby("Region")["Demand"]
      .agg(["mean", "sum", "count"])
      .sort_values("mean", ascending=False)
      .reset_index()
)

display(region_demand)


In [0]:
promotion_demand = (df.groupby("Promotion")["Demand"].agg(["mean", "sum", "count"]).reset_index())

display(promotion_demand)

In [0]:
print("DATA EXPLORATION SUMMARY")
print("====================================")

print("Rows              :", len(df))
print("Columns           :", len(df.columns))
print("Start Date        :", df["Date"].min())
print("End Date          :", df["Date"].max())
print("Stores            :", df["Store ID"].nunique())
print("Products          :", df["Product ID"].nunique())
print("Categories        :", df["Category"].nunique())
print("Regions           :", df["Region"].nunique())
print("Missing Values    :", df.isnull().sum().sum())
print("Duplicate Rows    :", df.duplicated().sum())
print("Zero Demand Rows  :", (df["Demand"] == 0).sum())

print("====================================")
print("Data exploration completed")
print("====================================")